02: “Improved: negations kept, one clean() function, class_weight='balanced'. Accuracy 88.5%, macro F1 0.85. Love recall 0.60 → 0.91, surprise recall 0.47 → 0.86 (precision dropped 0.90 → 0.76 and 0.88 → 0.66).”

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
import string

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
from google.colab import files
uploaded = files.upload()

Saving train.txt to train.txt


In [4]:
df = pd.read_csv('train.txt', sep=';', header=None, names=['text', 'emotion'])

In [5]:
df['emotion'].value_counts(normalize=True).round(3)

,proportion
emotion,
joy,0.335
sadness,0.292
anger,0.135
fear,0.121
love,0.082
surprise,0.036


In [6]:
df.head(4)

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love


In [7]:
df.isnull().sum()

,0
text,0
emotion,0


In [8]:
unique_emotions = df['emotion'].unique()
emotions_numbers = {}
i = 0
for emo in unique_emotions:
  i +=1
  emotions_numbers[emo] = i
print(i)



6


In [9]:
df

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [10]:
df

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [11]:
stop_words = set(stopwords.words('english'))
keep = {w for w in stop_words if w in {'no', 'not', 'nor'} or "n't" in w}
stop_words = stop_words - keep

In [12]:
len(stop_words)

177

In [13]:
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

def remove_num(txt):
  new = ""
  for i in txt:
    if not i.isdigit():
      new = new + i
  return new

def remove_emoji(txt):
  rm_emoji = ""
  for i in txt:
    if i.isascii():
      rm_emoji = rm_emoji + i
  return rm_emoji

def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)
  return ' '.join(cleaned)

In [14]:
def clean(txt):
    txt = txt.lower()
    txt = remove(txt)
    txt = remove_punc(txt)
    txt = remove_num(txt)
    txt = remove_emoji(txt)
    return txt

df['text'] = df['text'].apply(clean)

In [15]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['emotion'], test_size=0.20, random_state=42)

In [16]:
df.head(4)

,text,emotion
0,didnt feel humiliated,sadness
1,go feeling hopeless damned hopeful around some...,sadness
2,im grabbing minute post feel greedy wrong,anger
3,ever feeling nostalgic fireplace know still pr...,love


In [17]:
df.shape

(16000, 2)

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer , CountVectorizer

In [19]:
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

tfidf_vectorizer = TfidfVectorizer()

In [20]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [21]:
nb_model = MultinomialNB()

In [22]:
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

nb_model.fit(X_train_bow, y_train)
nb_model

MultinomialNB()

In [23]:
pred_nb = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_nb))

0.766875


In [24]:
pred_nb

array(['sadness', 'joy', 'sadness', ..., 'joy', 'joy', 'sadness'],
      dtype='<U8')

In [25]:
y_test

,emotion
8756,sadness
4660,joy
6095,sadness
304,joy
8241,sadness
...,...
15578,joy
5746,joy
6395,joy
7624,joy


In [26]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

tfidf_vectorizer = TfidfVectorizer()

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [27]:
y_pred = nb2_model.predict(X_test_tfidf)

In [28]:
print(accuracy_score(y_test, y_pred))

0.65875


In [29]:
from sklearn.linear_model import LogisticRegression

In [30]:
logistic_model  = LogisticRegression(max_iter=1000, class_weight='balanced')

In [31]:
logistic_model.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [32]:
log_pred = logistic_model.predict(X_test_tfidf)

In [33]:
print(accuracy_score(y_test, log_pred))

0.8846875


In [34]:
from sklearn.pipeline import Pipeline
model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
model.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer()),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [35]:
log_pred = model.predict(X_test)
print(accuracy_score(y_test, log_pred))

0.8846875


In [36]:
from sklearn.metrics import classification_report
print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

       anger       0.87      0.90      0.89       427
        fear       0.84      0.82      0.83       397
         joy       0.93      0.89      0.91      1021
        love       0.76      0.91      0.83       296
     sadness       0.95      0.89      0.92       946
    surprise       0.66      0.86      0.74       113

    accuracy                           0.88      3200
   macro avg       0.83      0.88      0.85      3200
weighted avg       0.89      0.88      0.89      3200



In [37]:
import joblib, sklearn
joblib.dump(model, 'emotion_model.joblib')
with open('stop_words.txt', 'w') as f:
    f.write('\n'.join(sorted(stop_words)))
print(sklearn.__version__)          # ye version mujhe batana

from google.colab import files
files.download('emotion_model.joblib')
files.download('stop_words.txt')

1.6.1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>